# 03 - Data Preprocessing

"Cleaning" means making the data trustworthy before modelling. Three checks:

1. **Duplicate rows** — is the same borrower listed twice?
2. **Useless columns** — like the ID, which carries no real information.
3. **"Cheating" columns (leakage)** — a column that secretly already reveals the answer (correlation to `Default` above 0.90 would be suspicious).

**Input:** `../01-data-collection/data/Loan_default.csv`
**Output:** `data/cleaned_data.csv`


In [1]:
import os
import numpy as np
import pandas as pd

DATA_PATH = "../01-data-collection/data/Loan_default.csv"
TARGET = "Default"
DROP_COLUMNS = ["LoanID"]
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Starting rows: {len(df):,}")

# Check 1: duplicate rows
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")
if n_dupes:
    df = df.drop_duplicates()
    print(f"Removed. {len(df):,} rows left.")


Starting rows: 255,347


Duplicate rows: 0


In [2]:
# Check 2: any column that looks suspiciously like "cheating"?
print("How strongly does each number-column match Default?")
numeric_all = df.select_dtypes(include=[np.number])
corr_to_target = (numeric_all.corr()[TARGET].drop(TARGET)
                  .sort_values(key=abs, ascending=False))
for col, r in corr_to_target.items():
    flag = "  <-- TOO STRONG, investigate!" if abs(r) > 0.90 else ""
    print(f"   {col:<20} {r:+.3f}{flag}")

if corr_to_target.abs().max() < 0.90:
    print("\nGood - nothing is suspiciously strong. No cheating columns found.")


How strongly does each number-column match Default?
   Age                  -0.168
   InterestRate         +0.131
   Income               -0.099
   MonthsEmployed       -0.097
   LoanAmount           +0.087
   CreditScore          -0.034
   NumCreditLines       +0.028
   DTIRatio             +0.019
   LoanTerm             +0.001

Good - nothing is suspiciously strong. No cheating columns found.


In [3]:
# Check 3: drop the useless ID column
df = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns])
print(f"Dropped: {DROP_COLUMNS}")
print(f"Data ready: {df.shape[0]:,} rows, {df.shape[1]} columns")

df.to_csv(os.path.join(OUTPUT_DIR, "cleaned_data.csv"), index=False)
print("Saved: data/cleaned_data.csv")


Dropped: ['LoanID']
Data ready: 255,347 rows, 17 columns


Saved: data/cleaned_data.csv


## How to read this

- **Zero duplicate rows** — nothing to remove.
- **The strongest match is `Age` at only -0.168** — nowhere near the 0.90 danger line, so there is no "cheating" column hiding in this dataset.
- We dropped `LoanID`, leaving **17 columns** in `data/cleaned_data.csv`.

## Next module
Module 4 (Feature Engineering) turns the remaining text columns into numbers and splits the data into training and test sets.
